In [74]:
import numpy as np
import pandas as pd

In [75]:
df = pd.read_csv("IMDB Dataset.csv")

In [76]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [77]:
df.shape

(50000, 2)

In [78]:
df['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [79]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [80]:
import re
from nltk.corpus import stopwords

In [81]:
df['y'] = (df['sentiment'] == 'positive').astype(int)

In [82]:
# Don't remove this stop words as we require them to analyze sentiment if we remove them then the semantic meaning of the sentence would change
keep = {'not', 'no', 'nor', 'never', 'very', 'too'}
stop = set(stopwords.words('english')) - keep

def clean(t):
    t = re.sub(r'<.*?>', ' ', t.lower())
    t = re.sub(r'[^a-z\' ]', ' ', t)
    return ' '.join(w for w in t.split() if w not in stop)

df['clean'] = df['review'].apply(clean)

In [83]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split( df['clean'], df['y'], test_size=0.2, random_state=42, stratify=df['y'])

In [84]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.9, sublinear_tf=True)

In [85]:
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

In [86]:
X_train_vec.shape

(40000, 283999)

In [87]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score

In [88]:
models = {
    'naive_bayes': MultinomialNB(),
    'logistic_regression': LogisticRegression(max_iter=1000),
    'svm': LinearSVC()
}

results = []

for name, model in models.items():
    model.fit(X_train_vec, y_train)
    p = model.predict(X_test_vec)
    results.append((name, accuracy_score(y_test, p), f1_score(y_test, p)))

In [89]:
print(pd.DataFrame(results, columns=['model', 'accuracy', 'f1']))

                 model  accuracy        f1
0          naive_bayes    0.8939  0.894207
1  logistic_regression    0.9076  0.908333
2                  svm    0.9154  0.915771
